# 02 — Data Preprocessing

**Purpose:** Clean and prepare the TMDB dataset for all three recommender algorithms.

**Input:** Raw CSV files in `data/raw/`
**Output:** Cleaned CSV files in `data/processed/`

**Steps:**
1. Load raw data
2. Drop movies with no ratings (cold-start items)
3. Deduplicate movies with the same tmdbId
4. Parse JSON columns (genres, keywords) into flat lists
5. Compute per-movie rating statistics
6. Merge movies + ratings + credits into unified tables
7. Save processed outputs

**References:**
- Schein et al. (2002) — cold-start problem
- Ilyas & Chu (2015) — data deduplication
- Adomavicius & Zhang (2012) — rating data characteristics
- Lops et al. (2010) — content-based feature extraction
- Koren et al. (2009) — matrix factorization for recommender systems

## 1. Load Libraries and Raw Data

In [ ]:
import pandas as pd
import json
import os

In [ ]:
# Load the three raw datasets.
# We never modify the originals — all changes happen on copies.
movies_raw = pd.read_csv('../data/raw/tmdb_movie_dataset.csv')
ratings_raw = pd.read_csv('../data/raw/tmdb_movie_ratings.csv')
credits_raw = pd.read_csv('../data/raw/tmdb_movie_credits.csv')

print(f'Loaded: movies={movies_raw.shape}, ratings={ratings_raw.shape}, credits={credits_raw.shape}')

## 2. Drop Movies With No Ratings (Cold-Start Items)

**What we do:** Remove 7 movies whose `ratingId` does not appear in the ratings file.

**Why (literature):** Items with zero interactions are called *cold-start items*. Collaborative filtering requires user-item interaction history to compute similarity or learn latent factors. Items with no ratings provide no signal for any user-based or model-based CF approach and would produce unreliable recommendations.

> Schein, A. I., Popescul, A., Ungar, L. H., & Pennock, D. M. (2002). Methods and metrics for cold-start recommendations. *Proceedings of the 25th Annual International ACM SIGIR Conference on Research and Development in Information Retrieval*, 253–260.
>
> Lika, B., Kolomvatsos, K., & Hadjiefthymiades, S. (2014). Facing the cold start problem in recommender systems. *Expert Systems with Applications*, 41(4), 2065–2073.

Both papers identify the cold-start problem as a fundamental challenge: the system cannot generate meaningful recommendations for items (or users) with no recorded interactions. Dropping unrated items is the standard first step before building any collaborative model.

In [ ]:
# Find which ratingIds exist in the ratings file
valid_rating_ids = set(ratings_raw['ratingId'])

# Keep only movies whose ratingId appears in the ratings table
movies = movies_raw[movies_raw['ratingId'].isin(valid_rating_ids)].copy()

dropped = len(movies_raw) - len(movies)
print(f'Dropped {dropped} cold-start movies (no user ratings)')
print(f'Movies remaining: {len(movies)}')

## 3. Deduplicate Movies by tmdbId

**What we do:** 4 movies appear twice with the same `tmdbId` but different `ratingId`. We keep the entry with the most ratings.

**Why (literature):** Duplicate records violate data integrity and can bias model training by over-representing certain items. In recommender systems, duplicated item entries cause the same movie to appear multiple times in the user-item matrix, distorting similarity computations.

> Ilyas, I. F., & Chu, X. (2015). Trends in cleaning relational data: Consistency and deduplication. *Foundations and Trends in Databases*, 5(4), 283–399.

The authors establish that deduplication is a critical data quality step before any downstream analysis. We resolve duplicates by keeping the entry with the richer rating history, as it provides more collaborative signal.

In [ ]:
# Count how many ratings each ratingId has
rating_counts = ratings_raw.groupby('ratingId').size().reset_index(name='num_ratings')

# Merge with movies to see which entry has more ratings
movies_with_counts = movies.merge(rating_counts, on='ratingId', how='left')

# For duplicate tmdbIds, keep the row with the higher num_ratings
movies_deduped = movies_with_counts.sort_values('num_ratings', ascending=False).drop_duplicates(subset='tmdbId', keep='first')

# Drop the helper column
movies_deduped = movies_deduped.drop(columns=['num_ratings'])

print(f'Before dedup: {len(movies)} rows, {movies["tmdbId"].nunique()} unique tmdbId')
print(f'After dedup:  {len(movies_deduped)} rows, {movies_deduped["tmdbId"].nunique()} unique tmdbId')
movies = movies_deduped.reset_index(drop=True)

## 4. Parse JSON Columns (Genre and Keyword Feature Extraction)

**What we do:** Convert the JSON-formatted `genres` and `keywords` columns into Python lists and pipe-delimited strings.

**Why (literature):** Content-based and hybrid recommender systems rely on item metadata (genres, keywords, descriptions) to model item characteristics. These features must be extracted from raw storage formats into structured representations before they can be used.

> Lops, P., De Gemmis, M., & Semeraro, G. (2010). Content-based recommender systems: State of the art and trends. In P. B. (Ed.), *Recommender Systems Handbook* (pp. 73–105). Springer.

The authors describe how content-based systems work by "matching up the attributes of a user profile" with "the attributes of a content object (item)." The genre and keyword fields in the TMDB dataset are these content attributes. Parsing them into lists enables:
- Mood-to-genre mapping (the core of our project)
- TF-IDF or one-hot encoding for content-based filtering
- Genre overlap as a similarity measure

In [ ]:
def parse_json_names(json_str):
    """Parse a JSON string containing {"id": ..., "name": ...} objects.
    
    Returns a list of name strings, e.g. ['Comedy', 'Crime'].
    Returns empty list if parsing fails.
    """
    try:
        items = json.loads(json_str)
        return [item['name'] for item in items]
    except (json.JSONDecodeError, TypeError, KeyError):
        return []

In [ ]:
# Parse genres into a list of genre names
# e.g. '[{"id": 35, "name": "Comedy"}]' -> ['Comedy']
movies['genre_list'] = movies['genres'].apply(parse_json_names)

# Parse keywords into a list of keyword names
movies['keyword_list'] = movies['keywords'].apply(parse_json_names)

# Create a pipe-delimited string for genres (useful for some algorithms)
movies['genres_str'] = movies['genre_list'].apply(lambda x: '|'.join(x))

# Verify the parsing worked
print('=== Sample parsed genres ===')
print(movies[['title', 'genre_list', 'genres_str']].head(5).to_string())
print()
print('=== Sample parsed keywords ===')
print(movies[['title', 'keyword_list']].head(3).to_string())

## 5. Compute Per-Movie Rating Statistics

**What we do:** Calculate `avg_rating`, `num_ratings`, and `rating_std` for each movie from the ratings table.

**Why (literature):** Rating frequency and value distributions significantly impact recommender system performance. Understanding these characteristics helps with:
- Filtering low-quality items (too few ratings = unreliable average)
- Identifying rating bias (some items have skewed distributions)
- Setting thresholds for minimum support in collaborative filtering

> Adomavicius, G., & Zhang, J. (2012). Impact of data characteristics on recommender systems performance. *ACM Transactions on Management Information Systems*, 3(1), 1–23.

The authors show that rating frequency distribution and rating value distribution are key data characteristics that affect algorithm accuracy. Computing these statistics early allows us to make informed filtering decisions later.

In [ ]:
# Calculate aggregate stats per movie from the ratings table
movie_stats = ratings_raw.groupby('ratingId').agg(
    avg_rating=('rating', 'mean'),       # average user rating
    num_ratings=('rating', 'count'),     # total number of ratings
    rating_std=('rating', 'std')         # standard deviation of ratings
).reset_index()

# Round for readability
movie_stats['avg_rating'] = movie_stats['avg_rating'].round(2)
movie_stats['rating_std'] = movie_stats['rating_std'].round(2)

# Merge stats into the movies dataframe
movies = movies.merge(movie_stats, on='ratingId', how='left')

print('=== Movie stats sample ===')
print(movies[['title', 'avg_rating', 'num_ratings', 'rating_std']].head(5).to_string())

## 6. Merge All Tables Into a Unified Dataset

**What we do:** Combine movies, credits, and rating statistics into a single `movies_full` table. Also enrich the ratings table with movie titles and genres.

**Why:** Having a denormalised dataset avoids repeated joins during algorithm development. All teammates load one file instead of reconstructing relationships.

In [ ]:
# Merge credits into movies on tmdbId
movies_full = movies.merge(
    credits_raw[['tmdbId', 'cast', 'crew']],
    on='tmdbId',
    how='left'
)

print(f'movies_full shape: {movies_full.shape}')
print(f'Columns: {movies_full.columns.tolist()}')

In [ ]:
# Enrich ratings with movie titles and genres for readability
ratings_full = ratings_raw.merge(
    movies[['ratingId', 'tmdbId', 'title', 'genres_str', 'genre_list']],
    on='ratingId',
    how='left'
)

print(f'ratings_full shape: {ratings_full.shape}')
print(f'Columns: {ratings_full.columns.tolist()}')
print()
print('=== Sample enriched ratings ===')
print(ratings_full.head(5).to_string())

## 7. User-Item Matrix Size and Sparsity

**What we do:** Compute the dimensions of the user-item matrix and its sparsity.

**Why (literature):** Sparsity is one of the most fundamental challenges in collaborative filtering. The ratio of observed to total possible ratings determines which algorithms are feasible.

> Koren, Y., Bell, R., & Volinsky, C. (2009). Matrix factorization techniques for recommender systems. *Computer*, 42(8), 30–37.

The authors note that real-world rating matrices are extremely sparse (often >99%), which motivates the use of matrix factorization over neighbourhood-based methods. Matrix factorization handles sparsity by learning latent factors from the observed entries only.

In [ ]:
n_users = ratings_full['userId'].nunique()
n_movies = ratings_full['ratingId'].nunique()
n_ratings = len(ratings_full)
sparsity = 1 - n_ratings / (n_users * n_movies)

print(f'User-item matrix dimensions: {n_users:,} users x {n_movies:,} movies')
print(f'Total possible cells:        {n_users * n_movies:,}')
print(f'Observed ratings:            {n_ratings:,}')
print(f'Sparsity:                    {sparsity:.4%}')
print()
print('The matrix is over 99.7% empty — matrix factorization (SVD) is the')
print('appropriate approach for collaborative filtering at this scale.')

## 8. Final Validation

In [ ]:
# Validate the processed data before saving
print('=== movies_full validation ===')
print(f'Rows: {len(movies_full)}')
print(f'Unique tmdbId: {movies_full["tmdbId"].nunique()}')
print(f'Unique ratingId: {movies_full["ratingId"].nunique()}')
print(f'Missing avg_rating: {movies_full["avg_rating"].isnull().sum()}')
print(f'Empty genre_list: {(movies_full["genre_list"].apply(len) == 0).sum()}')
print()

print('=== ratings_full validation ===')
print(f'Rows: {len(ratings_full)}')
print(f'Unique users: {ratings_full["userId"].nunique()}')
print(f'Unique movies: {ratings_full["ratingId"].nunique()}')
print(f'Missing titles: {ratings_full["title"].isnull().sum()}')
print()

# Verify no duplicate rows
print(f'Duplicate rows in movies_full: {movies_full.duplicated().sum()}')
print(f'Duplicate rows in ratings_full: {ratings_full.duplicated().sum()}')

In [ ]:
# Show final column lists
print('=== movies_full columns ===')
print(movies_full.columns.tolist())
print()
print('=== ratings_full columns ===')
print(ratings_full.columns.tolist())

## 9. Save Processed Data

Save to `data/processed/` so all teammates can load the clean data directly.

In [ ]:
# Create the output directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save movies_full (movie metadata + parsed genres/keywords + rating stats + credits)
movies_full.to_csv('../data/processed/movies_clean.csv', index=False)
print(f'Saved movies_clean.csv: {movies_full.shape}')

# Save ratings_full (all ratings enriched with movie title and genres)
ratings_full.to_csv('../data/processed/ratings_clean.csv', index=False)
print(f'Saved ratings_clean.csv: {ratings_full.shape}')

# Save a simple movie lookup table (minimal columns for quick reference)
movie_lookup = movies[['ratingId', 'tmdbId', 'title', 'genres_str', 'genre_list', 'avg_rating', 'num_ratings']].copy()
movie_lookup.to_csv('../data/processed/movie_lookup.csv', index=False)
print(f'Saved movie_lookup.csv: {movie_lookup.shape}')

## 10. Preprocessing Summary

| Step | Action | Justification | Reference |
|---|---|---|---|
| Drop unrated movies | Removed 7 movies with no ratings | Cold-start items have no collaborative signal | Schein et al. (2002); Lika et al. (2014) |
| Deduplicate tmdbId | Kept entry with most ratings per tmdbId | Duplicate records distort similarity computation | Ilyas & Chu (2015) |
| Parse genres/keywords | Converted JSON to lists | Content attributes needed for feature extraction | Lops et al. (2010) |
| Compute rating stats | avg_rating, num_ratings, rating_std | Rating distributions affect algorithm performance | Adomavicius & Zhang (2012) |
| Merge tables | Denormalised into single files | Avoids repeated joins; shared by all teammates | — |
| Sparsity analysis | Computed user-item matrix sparsity | Motivates matrix factorization over neighbourhood CF | Koren et al. (2009) |

### Output Files

| File | Use Case |
|---|---|
| `movies_clean.csv` | Content-based filtering, feature extraction |
| `ratings_clean.csv` | Collaborative filtering, evaluation |
| `movie_lookup.csv` | Quick reference, UI display |

### References

1. Schein, A. I., Popescul, A., Ungar, L. H., & Pennock, D. M. (2002). Methods and metrics for cold-start recommendations. *Proceedings of the 25th Annual International ACM SIGIR Conference on Research and Development in Information Retrieval*, 253–260.

2. Lika, B., Kolomvatsos, K., & Hadjiefthymiades, S. (2014). Facing the cold start problem in recommender systems. *Expert Systems with Applications*, 41(4), 2065–2073.

3. Ilyas, I. F., & Chu, X. (2015). Trends in cleaning relational data: Consistency and deduplication. *Foundations and Trends in Databases*, 5(4), 283–399.

4. Lops, P., De Gemmis, M., & Semeraro, G. (2010). Content-based recommender systems: State of the art and trends. In P. B. (Ed.), *Recommender Systems Handbook* (pp. 73–105). Springer.

5. Adomavicius, G., & Zhang, J. (2012). Impact of data characteristics on recommender systems performance. *ACM Transactions on Management Information Systems*, 3(1), 1–23.

6. Koren, Y., Bell, R., & Volinsky, C. (2009). Matrix factorization techniques for recommender systems. *Computer*, 42(8), 30–37.